# 实验四：Vector RAG / GraphRAG / WikiRAG
使用 `myenv` 内核和真实的 `BAAI/bge-small-zh-v1.5` 嵌入模型，仅在 CPU 上运行。语料、问题、三元组、条目与全部检索逻辑位于同目录 `hw4_rag.py`。本 Notebook 直接运行该脚本并展示实验报告需要的证据；不需要 LLM API。

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import csv, json
from hw4_rag import OUT, QUESTIONS, run_all
print('问题总数:', len(QUESTIONS), '；内置 15 + 自拟 5')
print('输出目录:', OUT)

## 运行完整评测
会在 `lab04_outputs/` 中生成本实验的指标、逐题检索记录和图片，不改动其他实验结果。

In [ ]:
run_all()

## 报告表 1、表 3、表 4：配置、20 题指标、内置 15 题 K 扫描

In [ ]:
print(json.dumps(json.loads((OUT / 'config.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
for filename in ('metrics_20.csv', 'k_scan_15.csv', 'hybrid_metrics_20.csv'):
    print('\n', filename)
    with (OUT / filename).open(encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f): print(row)

## 图 1~3：Q01 检索排序、实体图、结果曲线

In [ ]:
for filename in ('vector_q01.png', 'graph.png', 'wiki_entries.png', 'recall_20.png', 'k_sensitivity_15.png'):
    print(filename)
    display(Image(filename=str(OUT / filename), width=800))

## 手算 MRR、失败案例和选做 RRF
Q01（多跳）、Q08（事实）、Q14（全局）选自 WikiRAG，首命中位置必须从真实检索记录核对。RRF 是选做实验；是否改进要以实际数据为准。

In [ ]:
print((OUT / 'mrr_manual_3.txt').read_text(encoding='utf-8'))
details = json.loads((OUT / 'retrieval_details.json').read_text(encoding='utf-8'))
for system in ('Vector RAG', 'GraphRAG', 'WikiRAG'):
    failures = [r for r in details if r['system'] == system and r['recall'] < 1]
    print(system, '严格漏检案例:', [(r['id'], r['evidence'], r['hits']) for r in failures])
    if len(failures) < 2: print('严格漏检不足 2 个；报告须注明，不能虚构。')

## 忠实度人工检查说明
本实验未配置可选 LLM API，因此不能把检索结果冒充为 LLM 生成答案。报告的忠实度栏需要你从每个范式的逐题 `scored_hits` 中选 3 题，写出**自己根据检索上下文形成的回答**，逐句核对所检索文档后手工打 1~5 分；依据不足时回答‘资料不足’。如果老师要求必须评价 LLM 生成答案，则还需先提供 API 并另做生成实验。